In [ ]:
#| default_exp genetic

This derivative-free optimiser minimises a scalar callable `objective(x, y)` using binary DNA, selection, one-point crossover and bit mutation. Fitness is exactly the negative objective: finite negative, positive and tied losses are all supported, without clipping or gradients.

The default **24-bit fixed-point encoding** quantises each coordinate into $2^{24}$ levels, spaced by its bound width divided by $2^{24}-1$. Every bit pattern decodes to a finite point in the rectangle. The optional **raw IEEE754 float64 encoding** instead stores the actual 64 bits of each coordinate: sign, 11-bit biased exponent, and 52-bit fraction, most-significant bit first. It is not fixed-point quantisation disguised as floating point. Its codec preserves signed zero, subnormals, extreme finite values, infinities and NaN payloads exactly; invalid search candidates are rejected separately, never clipped or repaired.

All modes share the same rectangle. By default, `optimize` supplies the objective to initialisation and the GA uses its `domain()` (coordinate-wise lower/upper pairs). Without a domain, it falls back to an offset of four around the start. An explicit `max_offset` overrides domain discovery and bounds **each coordinate** around the start throughout the run, not the Euclidean radius. The initial population is uniformly sampled there and includes the starting point (quantised only in fixed mode). A zero offset keeps the entire population at that exact point. Raw operators do **not** imply an unbounded search.

Each generation retains the best 30% unchanged, selects 40% proportionally to shifted fitness (fixed) or average fitness rank (IEEE754), and fills the remainder with two-way tournaments sampled with replacement. Counts round down, with at least one elite. Selected parents are shuffled and crossed in pairs to produce **two** children per pair; an unpaired parent is retained before mutation. Only non-elites mutate, so the best loss cannot worsen for a deterministic objective.

In [ ]:
#| exporti
from copy import deepcopy
from dataclasses import dataclass, field
from numbers import Integral, Real

import numpy as np

_BITS_PER_COORDINATE = 24
_MAX_GENE = (1 << _BITS_PER_COORDINATE) - 1
_BIT_SHIFTS = np.arange(_BITS_PER_COORDINATE - 1, -1, -1, dtype=np.uint32)
_BIT_WEIGHTS = np.uint32(1) << _BIT_SHIFTS
_FLOAT_SHIFTS = np.arange(63, -1, -1, dtype=np.uint64)
_FLOAT_WEIGHTS = np.uint64(1) << _FLOAT_SHIFTS

In [ ]:
#| export
@dataclass(frozen=True)
class Diagnostics:
    """Per-update costs; initial counters are zero, not cumulative.

    Offspring counts candidates checked, including retry attempts. Displacement
    is the mean absolute coordinate shift divided by bound width, measured from
    the shuffled pre-crossover parent to each accepted non-elite; fallbacks
    contribute zero. Zero-width coordinates contribute zero. Exponent counts
    cover both coordinates, with raw biased codes (0 includes subnormals).
    """

    evaluations: int = 0
    offspring: int = 0
    rejected_nonfinite: int = 0
    rejected_bounds: int = 0
    rejected_objective: int = 0
    retries: int = 0
    retained_parents: int = 0
    mutation_displacement: float = 0.0
    exponent_histogram: tuple = ()

    @property
    def rejected(self):
        return self.rejected_nonfinite + self.rejected_bounds + self.rejected_objective

    @property
    def rejection_rate(self):
        return self.rejected / self.offspring if self.offspring else 0.0


@dataclass(frozen=True)
class State:
    """A population snapshot: binary DNA, [lower, upper] bounds and private RNG state.

    Fitness is None until the first update, then the negative objective per individual.
    Treat arrays and the RNG dictionary as read-only; updates never modify them.
    """

    generation: np.ndarray
    bounds: np.ndarray
    rng_state: dict
    fitness: np.ndarray | None = None
    encoding: str = 'fixed'
    operators: str = 'standard'
    diagnostics: Diagnostics = field(default_factory=Diagnostics)

In [ ]:
#| exporti
def _encode(params, bounds):
    params = np.asarray(params, dtype=np.float64)
    width = bounds[1] - bounds[0]
    fractions = np.divide(params - bounds[0], width,
                          out=np.zeros_like(params), where=width != 0)
    genes = np.rint(np.clip(fractions, 0, 1) * _MAX_GENE).astype(np.uint32)
    return ((genes[..., None] >> _BIT_SHIFTS) & 1).reshape(-1, 48).astype(np.uint8)


def _decode(generation, bounds):
    genes = generation.reshape(-1, 2, _BITS_PER_COORDINATE) @ _BIT_WEIGHTS
    params = bounds[0] + (genes / _MAX_GENE) * (bounds[1] - bounds[0])
    return np.clip(params, bounds[0], bounds[1])


def _real_option(value, name, minimum, maximum=None):
    if isinstance(value, (bool, np.bool_)) or not isinstance(value, Real):
        raise ValueError(f'{name} must be a finite real number')
    try:
        value = float(value)
    except OverflowError as error:
        raise ValueError(f'{name} must be finite') from error
    if not np.isfinite(value) or value < minimum or (maximum is not None and value > maximum):
        limit = f'[{minimum}, {maximum}]' if maximum is not None else f'>= {minimum}'
        raise ValueError(f'{name} must be finite and {limit}')
    return value


def _starting_bounds(params, max_offset):
    try:
        params = np.asarray(params)
    except (TypeError, ValueError) as error:
        raise ValueError('params must contain exactly two finite real numbers') from error
    if params.shape != (2,) or params.dtype.kind not in 'iuf':
        raise ValueError('params must contain exactly two finite real numbers')
    with np.errstate(over='ignore', invalid='ignore'):
        params = params.astype(np.float64)
        bounds = np.array([params - max_offset, params + max_offset])
        width = bounds[1] - bounds[0]
    if not np.all(np.isfinite(params)):
        raise ValueError('params must contain exactly two finite real numbers')
    if (not np.all(np.isfinite(bounds)) or not np.all(np.isfinite(width))
            or (max_offset > 0 and np.any(width <= 0))):
        raise ValueError('max_offset must give finite, representable, nondegenerate bounds')
    return params, bounds


def _objective_bounds(params, objective):
    domain = getattr(objective, 'domain', None)
    if domain is None:
        return _starting_bounds(params, 4)
    params, _ = _starting_bounds(params, 0)
    bounds = np.asarray(domain())
    if bounds.shape != (2, 2) or bounds.dtype.kind not in 'iuf':
        raise ValueError('objective domain must contain two finite [lower, upper] pairs')
    with np.errstate(over='ignore', invalid='ignore'):
        bounds = bounds.astype(np.float64).T.copy()
        width = bounds[1] - bounds[0]
    if not np.all(np.isfinite(bounds)) or not np.all(np.isfinite(width)) or np.any(width < 0):
        raise ValueError('objective domain must have finite, ordered, representable bounds')
    if np.any(params < bounds[0]) or np.any(params > bounds[1]):
        raise ValueError('params must lie within the objective domain')
    return params, bounds

In [ ]:
#| export
def encode_float64(params):
    """Encode a real pair or (n, 2) array into (n, 128) MSB-first uint8 bits.

    Numeric inputs are converted to native float64, not reinterpreted as it.
    Float64 payloads are exact, including NaNs; search validation is separate.
    """
    params = np.asarray(params)
    if params.dtype.kind not in 'iuf' or not (
            params.shape == (2,) or (params.ndim == 2 and params.shape[1] == 2)):
        raise ValueError('params must be a real pair or an array of shape (n, 2)')
    native = np.ascontiguousarray(params, dtype=np.float64).reshape(-1, 2)
    words = native.view(np.uint64)
    return ((words[..., None] >> _FLOAT_SHIFTS) & 1).astype(np.uint8).reshape(len(native), 128)


def decode_float64(generation):
    """Decode binary (n, 128) DNA exactly, without rejecting or clipping IEEE values."""
    generation = np.asarray(generation)
    if (generation.ndim != 2 or generation.shape[1] != 128
            or generation.dtype.kind not in 'biuf'
            or not np.all((generation == 0) | (generation == 1))):
        raise ValueError('generation must be a binary array of shape (n, 128)')
    words = generation.astype(np.uint64).reshape(len(generation), 2, 64) @ _FLOAT_WEIGHTS
    return words.view(np.float64).copy()


def mutation_rates(encoding='fixed', operators='standard', mutation_chance=0.01):
    """Per-bit rates with sum genome_length * mutation_chance.

    Guarded sign/exponent/fraction weights are .05/.1/1. Water-filling caps
    saturated fields at one and redistributes the remaining mutation budget.
    Thus all rates are zero at p=0 and one at p=1, with no frozen fields.
    """
    if not isinstance(encoding, str) or encoding not in ('fixed', 'ieee754'):
        raise ValueError("encoding must be 'fixed' or 'ieee754'")
    if not isinstance(operators, str) or operators not in ('standard', 'guarded'):
        raise ValueError("operators must be 'standard' or 'guarded'")
    if encoding == 'fixed' and operators == 'guarded':
        raise ValueError("guarded operators require encoding='ieee754'")
    chance = _real_option(mutation_chance, 'mutation_chance', 0, 1)
    size = 48 if encoding == 'fixed' else 128
    if operators == 'standard' or chance in (0, 1):
        return np.full(size, chance)
    weights = np.tile(np.r_[0.05, np.full(11, 0.1), np.ones(52)], 2)
    rates = np.zeros(size)
    active = np.ones(size, dtype=bool)
    budget = size * chance
    while np.any(active):
        proposed = weights[active] * (budget / weights[active].sum())
        rates[active] = np.minimum(proposed, 1)
        saturated = active & (rates >= 1)
        if not np.any(saturated):
            break
        active[saturated] = False
        budget -= np.count_nonzero(saturated)
    return rates

In [ ]:
#| export
def get_params(state):
    """Decode a snapshot into a fresh float64 array of shape (population_size, 2)."""
    if state.encoding == 'ieee754':
        return decode_float64(state.generation)
    if state.encoding == 'fixed':
        return _decode(state.generation, state.bounds)
    raise ValueError("state encoding must be 'fixed' or 'ieee754'")

In [ ]:
#| exporti
def _objective_value(objective, x, y, allow_nonfinite=False):
    if allow_nonfinite:
        # Silence warning-only arithmetic for rejectable proposals, never 'raise'.
        quiet = {key: 'ignore' for key, mode in np.geterr().items() if mode == 'warn'}
        with np.errstate(**quiet):
            loss = np.asarray(objective(x, y))
    else:
        loss = np.asarray(objective(x, y))
    if loss.shape != () or loss.dtype.kind not in 'iuf':
        raise ValueError('objective(x, y) must return a finite real scalar')
    value = float(loss)
    if not allow_nonfinite and not np.isfinite(value):
        raise ValueError('objective(x, y) must return a finite real scalar')
    return value


def _evaluate(objective, params):
    if not callable(objective):
        raise ValueError('objective must be callable as objective(x, y)')
    fitness = np.empty(len(params), dtype=np.float64)
    for index, (x, y) in enumerate(params):
        fitness[index] = -_objective_value(objective, x, y)
    return fitness


def _proportional_probabilities(fitness):
    # Scale before subtracting: opposite, very large finite values must not overflow.
    scaled = fitness / max(1.0, np.max(np.abs(fitness)))
    weights = scaled - scaled.min()
    total = weights.sum()
    return weights / total if total > 0 else None


def _crossover(parents, rng, return_order=False):
    order = rng.permutation(len(parents))
    children = parents[order].copy()
    for first in range(0, len(children) - 1, 2):
        locus = rng.integers(1, children.shape[1])
        children[first:first + 2, locus:] = children[first:first + 2, locus:][::-1].copy()
    return (children, order) if return_order else children


def _rank_probabilities(fitness):
    _, inverse, counts = np.unique(fitness, return_inverse=True, return_counts=True)
    ranks = (np.cumsum(counts) - (counts - 1) / 2)[inverse]
    return ranks / ranks.sum()


def _select_parents(fitness, rng, ranked=False):
    count = len(fitness)
    elite_count = max(1, int(0.3 * count))
    proportional_count = int(0.4 * count)
    tournament_count = count - elite_count - proportional_count
    elites = np.argsort(-fitness, kind='stable')[:elite_count]
    probabilities = _rank_probabilities(fitness) if ranked else _proportional_probabilities(fitness)
    proportional = rng.choice(count, size=proportional_count, replace=True,
                              p=probabilities)
    competitors = rng.integers(count, size=(tournament_count, 2))
    winners = competitors[np.arange(tournament_count), np.argmax(fitness[competitors], axis=1)]
    return elites, np.concatenate([proportional, winners])


def _next_generation(generation, fitness, mutation_chance, rng, return_parents=False):
    elite_indices, selected = _select_parents(fitness, rng)
    elites, parents = generation[elite_indices], generation[selected]
    children, order = _crossover(parents, rng, return_order=True)
    children ^= rng.random(children.shape) < mutation_chance
    result = np.concatenate([elites, children])
    return (result, parents[order]) if return_parents else result

In [ ]:
#| exporti
def _guarded_crossover(parents, rng, return_order=False):
    order = rng.permutation(len(parents))
    children = parents[order].copy()
    for first in range(0, len(children) - 1, 2):
        swaps = rng.random((2, 3)) < 0.5
        for coordinate in range(2):
            for swap, (start, stop) in zip(swaps[coordinate], ((0, 1), (1, 12), (12, 64))):
                if swap:
                    block = slice(64 * coordinate + start, 64 * coordinate + stop)
                    children[first:first + 2, block] = children[first:first + 2, block][::-1].copy()
    return (children, order) if return_order else children


def _ieee_offspring(generation, selected, operators, rates, rng):
    crossover = _guarded_crossover if operators == 'guarded' else _crossover
    children, order = crossover(generation[selected], rng, return_order=True)
    children ^= rng.random(children.shape) < rates
    return children, selected[order]


def _exponent_histogram(generation):
    exponents = generation.reshape(-1, 64)[:, 1:12] @ (1 << np.arange(10, -1, -1))
    codes, counts = np.unique(exponents, return_counts=True)
    return tuple((int(code), int(count)) for code, count in zip(codes, counts))


def _displacement(before, after, bounds):
    width = bounds[1] - bounds[0]
    shifts = np.divide(np.abs(after - before), width,
                       out=np.zeros_like(after), where=width != 0)
    return float(np.mean(shifts))


def _explore_offspring(children, parents, bounds, chance, rng):
    explore = rng.random(len(children)) < chance
    count = np.count_nonzero(explore)
    points = rng.random((count, 2))
    sides = rng.random((count, 2))
    points = np.where(sides < 0.05, 0, np.where(sides > 0.95, 1, points))
    local = rng.random(count) < 0.5
    width = bounds[1] - bounds[0]
    origins = np.divide(parents[explore][local] - bounds[0], width,
                        out=np.zeros((np.count_nonzero(local), 2)), where=width != 0)
    scale = np.exp2(rng.uniform(-24, 0, size=(len(origins), 1)))
    # Sample local moves in unit coordinates: carry across float-bit cliffs,
    # reach closed boundaries, and avoid overflowing large physical ranges.
    points[local] = np.clip(origins + rng.normal(size=origins.shape) * scale, 0, 1)
    sampled = np.clip(bounds[0] + points * width, bounds[0], bounds[1])
    children[explore] = encode_float64(sampled)
    return children


def _next_ieee_generation(state, params, fitness, objective, operators, rates, max_retries, rng,
                          exploration_chance=0):
    def propose(selected):
        children, origins = _ieee_offspring(state.generation, selected, operators, rates, rng)
        if exploration_chance:
            children = _explore_offspring(children, params[origins], state.bounds, exploration_chance, rng)
        return children, origins

    elite_indices, selected = _select_parents(fitness, rng, ranked=True)
    children, origins = propose(selected)
    generation = np.concatenate([state.generation[elite_indices], children])
    next_fitness = np.empty(len(generation))
    next_fitness[:len(elite_indices)] = fitness[elite_indices]
    counts = dict(evaluations=len(params), offspring=0, rejected_nonfinite=0,
                  rejected_bounds=0, rejected_objective=0, retries=0, retained_parents=0)
    displacement = 0.0
    probabilities = _rank_probabilities(fitness)
    for slot, (child, origin) in enumerate(zip(children, origins), start=len(elite_indices)):
        for attempt in range(max_retries + 1):
            counts['offspring'] += 1
            candidate = decode_float64(child[None])[0]
            if not np.all(np.isfinite(candidate)):
                counts['rejected_nonfinite'] += 1
            elif np.any(candidate < state.bounds[0]) or np.any(candidate > state.bounds[1]):
                counts['rejected_bounds'] += 1
            else:
                counts['evaluations'] += 1
                value = _objective_value(objective, *candidate, allow_nonfinite=True)
                if np.isfinite(value):
                    generation[slot], next_fitness[slot] = child, -value
                    displacement += _displacement(params[origin], candidate, state.bounds)
                    break
                counts['rejected_objective'] += 1
            if attempt == max_retries:
                generation[slot], next_fitness[slot] = state.generation[origin], fitness[origin]
                counts['retained_parents'] += 1
            else:
                counts['retries'] += 1
                selected = rng.choice(len(params), size=2, replace=True, p=probabilities)
                proposals, retry_origins = propose(selected)
                child, origin = proposals[0], retry_origins[0]
    diagnostics = Diagnostics(**counts, mutation_displacement=displacement / len(children),
                              exponent_histogram=_exponent_histogram(generation))
    return generation, next_fitness, diagnostics

## Optimiser contract and controls

`genetic_algo(...)` returns `(init, update, get_params)`, compatible with `optimize(...).using(..., derivatives_based=False)`. `init([x, y], objective=None)` accepts two finite real coordinates, casts them to float64 and returns a `State`. `optimize` passes its function through the optional `init.with_objective` hook; direct callers can pass `objective` themselves for domain discovery. `update(i, objective, state)` returns a new snapshot whose fitness corresponds to its decoded population; `i` is accepted for optimiser compatibility. The objective is evaluated again on the input population, so changing the objective between updates does not reuse stale fitness or change the established bounds. The initial/input population must have finite real scalar objective values; otherwise updating raises `ValueError`.

- `population_size=50`: an integer of at least two; `None` preserves automatic sizing at 200 for two coordinates.
- `max_offset=None`: discover the objective domain, falling back to four when none is provided. An explicit finite, nonnegative offset overrides discovery; nonzero bounds must remain finite and distinct in float64. Domain bounds must be finite and ordered, and the start must lie inside them.
- `encoding='fixed'`: choose the original 48-bit fixed-point genome or `'ieee754'`, a 128-bit raw float64 genome.
- `operators='standard'`: uniform per-bit mutation and an arbitrary single crossover point across the entire genome. IEEE754 also supports `'guarded'`: independently exchange whole sign, exponent and fraction blocks in each coordinate, never mix exponent bits during crossover. Fixed + guarded is an error.
- `mutation_chance=0.01`: the mean per-bit mutation probability in `[0, 1]`. Standard rates are uniform. Guarded relative sign/exponent/fraction weights are `0.05/0.1/1`, scaled to the same expected `128 * mutation_chance` flips. Saturated rates are capped at one and their excess budget redistributed; at zero all rates are zero, at one all are one. `mutation_rates(...)` exposes the exact vector. Signs and exponents are never frozen, but this bias towards preserving scale is an experimental design choice, not a universally better optimiser. Fixed genomes have fewer bits, hence `48 * mutation_chance` expected flips at the same rate.
- `exploration_chance=None`: defaults to `0.1` in guarded mode and zero otherwise; nonzero values require guarded operators. After crossover/mutation, replace this fraction of non-elite proposals with bounded exploration samples, half global and half local around their selected parent. Global samples include each coordinate's endpoints with 5% probability per side; local Gaussian steps use log-uniform scales from $2^{-24}$ to one bound width and project onto the closed rectangle. This maintains global exploration and crosses float-bit cliffs during refinement, without altering elites or repairing invalid raw-bit proposals. Samples are encoded as exact float64 DNA and pass the same objective/retry checks. Set zero for the original bit-only operators; the per-bit mutation budget does not include these additional exploration moves.
- `max_retries=8`: a nonnegative integer limiting **additional** attempts per rejected IEEE754 offspring. Resample parents from the current evaluated population, then cross and mutate again. After exhaustion, retain a known valid parent with its exact cached fitness and report the fallback; never invent a score or silently repair a genome.
- `seed=None`: a NumPy random seed. A fixed seed gives reproducible populations and restarts; each state saves its own RNG snapshot, so replaying an update is deterministic and independent of other runs or the global NumPy RNG.

IEEE754 guardrails check nonfinite coordinates, then rectangle membership **before** calling the objective. Only a nonfinite objective **return** from an otherwise admissible new offspring causes rejection. Malformed/non-real/non-scalar returns and user exceptions always propagate, including explicit `FloatingPointError`. For these rejectable objective calls only, NumPy warning-mode arithmetic is locally silenced; caller `np.errstate(...='raise')` settings still raise. Fixed mode preserves its original strict finite-objective policy.

Each frozen `Diagnostics` snapshot reports actual `evaluations` (including parent re-evaluation), checked `offspring`, mutually exclusive rejection reasons, additional `retries`, and `retained_parents`. Fixed updates cost twice the population size; IEEE754 caches elite/fallback fitness but retries may cost more, so equal generation counts need not mean equal objective budgets. `rejected` sums the three reasons and `rejection_rate` divides by checked offspring. `mutation_displacement` is a finite mean absolute coordinate shift, normalised by each bound width, from shuffled pre-crossover parents to accepted non-elites; it includes crossover and exploration, counts fallbacks and zero-width coordinates as zero, and excludes elites. `exponent_histogram` counts both coordinates by raw biased exponent code (zero includes zeros/subnormals; 2047 never persists), or is empty for fixed mode.

The initial state has `fitness=None` and zero diagnostic counters (IEEE754 already has an exponent histogram). Updates do not mutate input arrays, previous generations, fitness or RNG dictionaries. `get_params` returns a fresh array, and history snapshots may safely be retained. As with other optimiser state objects, callers should not edit their internal arrays.

In [ ]:
#| export
def genetic_algo(population_size=50, max_offset=None, *, mutation_chance=0.01, seed=None,
                 encoding='fixed', operators='standard', max_retries=8, exploration_chance=None):
    """Return (init, update, get_params) for bounded, binary genetic minimisation.

    Use with derivatives_based=False. By default use the objective domain, or
    offset four if unavailable. Explicit max_offset bounds each coordinate
    around the starting point. Default fixed-point coordinates use 24
    bits; encoding='ieee754' uses exact float64 bits with standard or guarded
    operators. Invalid IEEE offspring retry at most max_retries additional
    times, then retain a valid parent. Input objective values must be finite;
    malformed returns and exceptions always propagate. Mutation rates have
    mean mutation_chance (see mutation_rates). Diagnostics measure each update.
    Guarded mode also explores globally and locally with probability 0.1;
    set exploration_chance=0 for bit-only proposals.
    A fixed seed makes initialisation and independent replays reproducible.
    """
    if population_size is None:
        population_size = 200
    if (isinstance(population_size, (bool, np.bool_))
            or not isinstance(population_size, Integral) or population_size < 2):
        raise ValueError('population_size must be an integer >= 2, or None')
    population_size = int(population_size)
    if max_offset is not None:
        max_offset = _real_option(max_offset, 'max_offset', 0)
    mutation_chance = _real_option(mutation_chance, 'mutation_chance', 0, 1)
    rates = mutation_rates(encoding, operators, mutation_chance)
    if exploration_chance is None:
        exploration_chance = 0.1 if operators == 'guarded' else 0
    exploration_chance = _real_option(exploration_chance, 'exploration_chance', 0, 1)
    if exploration_chance and operators != 'guarded':
        raise ValueError('nonzero exploration_chance requires guarded operators')
    if (isinstance(max_retries, (bool, np.bool_))
            or not isinstance(max_retries, Integral) or max_retries < 0):
        raise ValueError('max_retries must be an integer >= 0')
    max_retries = int(max_retries)
    initial_rng = np.random.default_rng(seed)
    bit_generator_type = type(initial_rng.bit_generator)
    initial_rng_state = deepcopy(initial_rng.bit_generator.state)

    def restore_rng(rng_state):
        rng = np.random.Generator(bit_generator_type(0))
        rng.bit_generator.state = deepcopy(rng_state)
        return rng

    def init(params, objective=None):
        params, bounds = (_objective_bounds(params, objective) if max_offset is None
                          else _starting_bounds(params, max_offset))
        rng = restore_rng(initial_rng_state)
        population = np.clip(bounds[0] + rng.random((population_size, 2)) * (bounds[1] - bounds[0]),
                             bounds[0], bounds[1])
        population[0] = params
        generation = encode_float64(population) if encoding == 'ieee754' else _encode(population, bounds)
        histogram = _exponent_histogram(generation) if encoding == 'ieee754' else ()
        return State(generation=generation, bounds=bounds, encoding=encoding, operators=operators,
                     rng_state=deepcopy(rng.bit_generator.state),
                     diagnostics=Diagnostics(exponent_histogram=histogram))

    def update(i, objective, state):
        if state.encoding != encoding or state.operators != operators:
            raise ValueError('state encoding/operators do not match this optimiser')
        if state.generation.shape != (population_size, len(rates)):
            raise ValueError('state generation shape does not match this optimiser')
        rng = restore_rng(state.rng_state)
        params = get_params(state)
        if (not np.all(np.isfinite(params)) or np.any(params < state.bounds[0])
                or np.any(params > state.bounds[1])):
            raise ValueError('input population must be finite and within its bounds')
        fitness = _evaluate(objective, params)
        bounds = state.bounds.copy()
        if encoding == 'ieee754':
            generation, next_fitness, diagnostics = _next_ieee_generation(
                state, params, fitness, objective, operators, rates, max_retries, rng, exploration_chance)
        else:
            generation, parents = _next_generation(
                state.generation, fitness, mutation_chance, rng, return_parents=True)
            decoded = _decode(generation, bounds)
            next_fitness = _evaluate(objective, decoded)
            diagnostics = Diagnostics(evaluations=2 * population_size, offspring=len(parents),
                mutation_displacement=_displacement(_decode(parents, bounds), decoded[-len(parents):], bounds))
        return State(generation=generation, bounds=bounds,
                     fitness=next_fitness, encoding=encoding, operators=operators, diagnostics=diagnostics,
                     rng_state=deepcopy(rng.bit_generator.state))

    init.with_objective = init
    return init, update, get_params

## Seeded optimisation

No JAX-compatible objective or derivative is needed. This NumPy sphere example searches the rectangle `[-1, 7] x [-7, 1]`. Inspect the best individual with `get_params`, rather than treating the entire population as a single point.

In [ ]:
from optimisations.optimizers import optimize

In [ ]:
def sphere(x, y):
    return np.square(x) + np.square(y)

run = (optimize(sphere)
       .using(genetic_algo(population_size=50, max_offset=4, seed=7),
              name='ga', derivatives_based=False)
       .start_from([3.0, -3.0]))
history = run.update(30)
population = get_params(history[-1])
best = population[np.argmax(history[-1].fitness)]
best, sphere(*best)

In [ ]:
#| include: false
losses = [min(sphere(x, y) for x, y in get_params(state)) for state in history]
assert len(history) == 31
assert np.all(np.diff(losses) <= 0)
assert losses[-1] < losses[0] * 0.1
assert history[0].fitness is None
assert all(get_params(state).shape == (50, 2) for state in history)
assert np.all(population >= [ -1, -7]) and np.all(population <= [7, 1])

## Raw floats, with and without representation-aware operators

A short, reproducible comparison records actual evaluation cost as well as best loss. This is an illustration, not a performance claim: raw exponent flips can jump many orders of magnitude, while guarded field bias often trades those jumps for smaller moves and supplements them with bounded exploration. Both still need rejection guardrails. On a sphere, very small coordinates can even make the computed squared loss underflow to zero; a reported zero is not proof of reaching the mathematical minimiser. Inspect diagnostics and compare multiple objectives, seeds and evaluation budgets before drawing conclusions.

In [ ]:
comparison = []
for encoding, operators in [('fixed', 'standard'), ('ieee754', 'standard'), ('ieee754', 'guarded')]:
    init, update, decode = genetic_algo(20, seed=7, encoding=encoding, operators=operators)
    state = init([3.0, -3.0])
    evaluations = 0
    for generation in range(15):
        state = update(generation, sphere, state)
        evaluations += state.diagnostics.evaluations
    comparison.append((encoding, operators, -state.fitness.max(), evaluations,
                       state.diagnostics.rejection_rate))
comparison

In [ ]:
#| include: false
#| eval: false
from matplotlib import rc_context

from optimisations.functions import himmelblau, eggholder
from optimisations.genetic import genetic_algo
from optimisations.optimizers import optimize
from optimisations.animations import animate

with rc_context({'animation.embed_limit': 100}):
    animate(
        optimize(eggholder())
            .using(
                genetic_algo(
                    encoding='ieee754',
                    operators='guarded',
                    mutation_chance=0.48 / 128,
                    seed=7,
                ),
                derivatives_based=False,
            )
            .start_from([0.0, 0.0]),
        frames=200,
        interval=300,
        output='js',
        show_diagnostics=True,
    );

In [8]:
#| eval: false

from optimisations.functions import eggholder
from optimisations.optimizers import optimize
from optimisations.genetic import genetic_algo


history = (optimize(eggholder())
            .using(
                genetic_algo(
                    encoding='ieee754',
                    operators='guarded',
                    mutation_chance=0.48 / 128,
                    seed=7,
                ),
                derivatives_based=False,
            )
            .start_from([0, 0])
            .update(1000)
 )

import numpy as np
from optimisations.genetic import get_params

best_index = np.argmax(history[-1].fitness)
get_params(history[-1])[best_index], -history[-1].fitness[best_index]
# Approximately (512, 404.2318), -959.6407.

(array([512.        , 404.23180472]), np.float64(-959.6406627208507))